# 01 — Generación de datos sintéticos

## Objetivo

Construir un portafolio sintético de seguros mediante un proceso generador de datos conocido (*Data Generating Process*, DGP) para comparar posteriormente modelos estadísticos clásicos, Machine Learning y Deep Learning bajo condiciones controladas.

El costo agregado de una póliza se modela mediante dos componentes: frecuencia (número de siniestros) y severidad (costo de cada siniestro).

$$N_i \mid X_i \sim \operatorname{Poisson}(\lambda_i),\qquad Z_{ij} \mid X_i \sim \operatorname{Lognormal}(\mu_i, \sigma^2).$$

Por tanto, la prima pura verdadera es:

$$E[S_i \mid X_i] = E[N_i \mid X_i]E[Z_i \mid X_i] = \lambda_i E[Z_i \mid X_i].$$

Las variables del proceso generador se conservan exclusivamente como *oracle* para evaluar los modelos; no se utilizarán como predictores.

## Configuración e imports

In [179]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
N = 10_000

rng = np.random.default_rng(SEED)

## 1. Variables explicativas

Se generan seis características observables de cada póliza. Estas constituirán el conjunto de predictores disponible para los modelos.

In [180]:
edad = rng.integers(18, 75, N)
zona = rng.integers(1, 6, N)  # 1: bajo riesgo; 5: muy alto riesgo
anios_vehiculo = rng.integers(0, 20, N)

cobertura = rng.choice([0, 1, 2], N, p=[0.50, 0.30, 0.20])
# 0: limitada; 1: responsabilidad civil; 2: amplia

historial = rng.choice([0, 1, 2, 3, 4], N, p=[0.50, 0.28, 0.14, 0.05, 0.03])
uso = rng.choice([0, 1, 2], N, p=[0.50, 0.42, 0.08]) # 0: particular; 1: transporte privado; 2: carga comercial

## 2. Frecuencia de siniestros

La frecuencia se genera con un modelo Poisson. `tuvo_siniestro` es una variable derivada de la frecuencia observada y `prob_siniestro` es la probabilidad teórica de al menos un siniestro.

In [181]:
eta_freq = (
    -2.8
    + 0.18 * zona
    + 0.30 * historial
    + 0.08 * cobertura
    + 0.015 * anios_vehiculo
    - 0.0005 * (edad - 40) ** 2
    + 0.10 * (uso == 2)
    + 0.05 * cobertura * uso
)

lambda_freq = np.exp(eta_freq)
numero_siniestros = rng.poisson(lambda_freq)
tuvo_siniestro = (numero_siniestros > 0).astype(int)
prob_siniestro = 1 - np.exp(-lambda_freq)

## 3. Severidad de siniestros

La severidad individual sigue una distribución Lognormal. La severidad promedio observada se deja como `NaN` cuando no existen siniestros, pues no está definida para esas pólizas.

\[E[Z_i \mid X_i] = \exp(\mu_i + \sigma^2 / 2).\]

In [182]:
mu_sev = (
    8.5
    + 0.10 * zona
    + 0.20 * cobertura
    + 0.012 * anios_vehiculo
    + 0.15 * (uso == 2)
    + 0.04 * cobertura * zona
)

sigma_sev = 0.55
expected_severity = np.exp(mu_sev + sigma_sev ** 2 / 2)

siniestro_total = np.zeros(N)
severidad_promedio = np.full(N, np.nan)

for i, n_claims in enumerate(numero_siniestros):
    if n_claims > 0:
        severidades = rng.lognormal(mean=mu_sev[i], sigma=sigma_sev, size=n_claims)
        siniestro_total[i] = severidades.sum()
        severidad_promedio[i] = severidades.mean()

## 4. Prima pura verdadera

La prima pura es la esperanza condicional del costo agregado, obtenida como frecuencia esperada por severidad esperada.

In [183]:
prima_pura_real = lambda_freq * expected_severity

## 5. DataFrame definitivo

In [184]:
df = pd.DataFrame({
    # Features observables
    'edad': edad,
    'zona': zona,
    'anios_vehiculo': anios_vehiculo,
    'cobertura': cobertura,
    'historial_siniestros': historial,
    'tipo_uso': uso,

    # Resultados observados
    'numero_siniestros': numero_siniestros,
    'tuvo_siniestro': tuvo_siniestro,
    'severidad_promedio': severidad_promedio,
    'costo_siniestros': siniestro_total,

    # Ground truth / oracle
    'lambda_real': lambda_freq,
    'prob_siniestro_real': prob_siniestro,
    'severidad_esperada_real': expected_severity,
    'prima_pura_real': prima_pura_real,
})

## 6. Roles de las variables

In [185]:
FEATURES = [
    'edad', 'zona', 'anios_vehiculo', 'cobertura',
    'historial_siniestros', 'tipo_uso',
]

TARGETS = [
    'numero_siniestros', 'severidad_promedio', 'costo_siniestros',
]

ORACLE = [
    'lambda_real', 'prob_siniestro_real',
    'severidad_esperada_real', 'prima_pura_real',
]

## 7. Validaciones y *sanity checks*

In [186]:
mask_claims = df['numero_siniestros'] > 0

print(f'Frecuencia esperada promedio: {lambda_freq.mean():.4f}')
print(f'Frecuencia observada promedio: {numero_siniestros.mean():.4f}')
print(f'Probabilidad teórica de siniestro: {prob_siniestro.mean():.2%}')
print(f'Pólizas con siniestro: {tuvo_siniestro.mean():.2%}')
print()
print(f"Severidad esperada promedio (con siniestro): ${df.loc[mask_claims, 'severidad_esperada_real'].mean():,.2f}")
print(f"Severidad observada promedio (con siniestro): ${df.loc[mask_claims, 'severidad_promedio'].mean():,.2f}")
print()
pp_media = df['prima_pura_real'].mean()
costo_medio = df['costo_siniestros'].mean()
print(f'Prima pura verdadera promedio: ${pp_media:,.2f}')
print(f'Costo observado promedio: ${costo_medio:,.2f}')
print(f'Diferencia relativa: {costo_medio / pp_media - 1:.2%}')

Frecuencia esperada promedio: 0.1602
Frecuencia observada promedio: 0.1554
Probabilidad teórica de siniestro: 14.53%
Pólizas con siniestro: 14.23%

Severidad esperada promedio (con siniestro): $12,442.29
Severidad observada promedio (con siniestro): $12,422.47

Prima pura verdadera promedio: $2,007.02
Costo observado promedio: $1,951.32
Diferencia relativa: -2.78%


## 8. Assertions

In [187]:
assert df[FEATURES].isna().sum().sum() == 0
assert (df['numero_siniestros'] >= 0).all()
assert (df['costo_siniestros'] >= 0).all()
assert (df['lambda_real'] > 0).all()
assert (df['severidad_esperada_real'] > 0).all()
assert (df['prima_pura_real'] > 0).all()
assert (df.loc[df['numero_siniestros'] == 0, 'costo_siniestros'] == 0).all()
assert df.loc[df['numero_siniestros'] == 0, 'severidad_promedio'].isna().all()
assert df.loc[df['numero_siniestros'] > 0, 'severidad_promedio'].notna().all()
assert (df.loc[df['numero_siniestros'] > 0, 'costo_siniestros'] > 0).all()

print('Todas las validaciones del DGP fueron superadas.')

Todas las validaciones del DGP fueron superadas.


## 9. Inspección final

Los gráficos y el análisis exploratorio se desarrollarán en `02_eda.ipynb`.

In [188]:
display(df.head())
df.info()
display(df[FEATURES + TARGETS + ORACLE].describe())

,edad,zona,anios_vehiculo,cobertura,historial_siniestros,tipo_uso,numero_siniestros,tuvo_siniestro,severidad_promedio,costo_siniestros,lambda_real,prob_siniestro_real,severidad_esperada_real,prima_pura_real
0,23,2,0,0,3,0,0,0,NaN,0.0,0.185537,0.169342,6983.112407,1295.626965
1,62,2,14,0,4,0,0,0,NaN,0.0,0.280271,0.244421,8260.579323,2315.196861
2,55,3,4,0,2,0,0,0,NaN,0.0,0.180414,0.165076,8097.008893,1460.815324
3,43,4,14,2,0,1,0,0,NaN,0.0,0.198990,0.180442,20728.192310,4124.705903
4,42,2,11,0,1,0,0,0,NaN,0.0,0.138484,0.129323,7968.487662,1103.508579


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   edad                     10000 non-null  int64  
 1   zona                     10000 non-null  int64  
 2   anios_vehiculo           10000 non-null  int64  
 3   cobertura                10000 non-null  int64  
 4   historial_siniestros     10000 non-null  int64  
 5   tipo_uso                 10000 non-null  int64  
 6   numero_siniestros        10000 non-null  int64  
 7   tuvo_siniestro           10000 non-null  int64  
 8   severidad_promedio       1423 non-null   float64
 9   costo_siniestros         10000 non-null  float64
 10  lambda_real              10000 non-null  float64
 11  prob_siniestro_real      10000 non-null  float64
 12  severidad_esperada_real  10000 non-null  float64
 13  prima_pura_real          10000 non-null  float64
dtypes: float64(6), int64(8)
memory usa

,edad,zona,anios_vehiculo,cobertura,historial_siniestros,tipo_uso,numero_siniestros,severidad_promedio,costo_siniestros,lambda_real,prob_siniestro_real,severidad_esperada_real,prima_pura_real
count,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,1423.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,45.794700,2.98100,9.571000,0.709500,0.825500,0.593000,0.155400,12422.470478,1951.318490,0.160179,0.145288,11578.747415,2007.023224
std,16.377636,1.41012,5.756846,0.784073,1.031482,0.639994,0.399082,8742.437938,6318.307758,0.081941,0.065058,4129.840591,1635.607690
min,18.000000,1.00000,0.000000,0.000000,0.000000,0.000000,0.000000,1305.626001,0.000000,0.040844,0.040021,6318.581401,258.074938
25%,32.000000,2.00000,5.000000,0.000000,0.000000,0.000000,0.000000,6469.791646,0.000000,0.105084,0.099751,8632.159634,976.280556
50%,46.000000,3.00000,10.000000,1.000000,1.000000,1.000000,0.000000,10135.257484,0.000000,0.140191,0.130808,10375.992467,1498.919855
75%,60.000000,4.00000,15.000000,1.000000,1.000000,1.000000,0.000000,15947.000784,0.000000,0.190234,0.173234,13456.936810,2462.050947
max,74.000000,5.00000,19.000000,2.000000,4.000000,2.000000,4.000000,90380.891559,117785.073658,0.800915,0.551082,30249.955730,18375.067545


## 10. Exportación

Se guarda la versión reproducible del portafolio para los notebooks posteriores.

In [189]:
DATA_PATH = Path('../data/raw')
DATA_PATH.mkdir(parents=True, exist_ok=True)

output_path = DATA_PATH / 'synthetic_insurance_portfolio.csv'
df.to_csv(output_path, index=False)

print(f'Dataset guardado correctamente: {output_path}')

Dataset guardado correctamente: ..\data\raw\synthetic_insurance_portfolio.csv
